# Follow-up: `complaints.affected_product_id` owner mismatch (100%)

The main `complaints` deep dive found that **every single complaint with a populated `affected_product_id` (44,570/44,570) points to a product owned by someone other than the complaint's own `customer_id`.** That's `checks.py`'s existing `product_owner_mismatch` check, reproduced directly and confirmed.

Before concluding anything, this notebook checks four things that would distinguish "real synthetic-generation bug" from "actually explainable":

1. Does the **same check** on `transactions`/`digital_events` (the other two tables `relationship_checks()` runs this on) also show a high mismatch rate, or is `complaints` uniquely broken?
2. When a complaint's own `customer_id` **does** have products of their own, does `affected_product_id` ever land on one of THEIR OWN products, or is it always a stranger's -- even when a correct match was available?
3. Is the mismatch **temporally implausible** too (e.g. does the "wrong" product even exist yet as of the complaint's `creation_date`), which would rule out any story where the product ID just refers to something opened later?
4. Does the mismatch look **structured** (e.g. same offset every time, clustered on specific customers) or **uniformly random** across the whole `products` table -- structure would suggest a join/shuffle bug with a specific mechanism; uniform randomness points more toward `affected_product_id` being generated independently of `customer_id` altogether.

Run top to bottom.

In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    return con.execute(sql, params or []).df()

## 1. Same `product_owner_mismatch` check reproduced for `transactions` and `digital_events` -- is this complaints-specific?

In [2]:
for table, product_field in [("transactions", "product_id"), ("digital_events", "product_id")]:
    result = q(f"""
        SELECT
            count(*) AS total_linked,
            count(*) FILTER (WHERE c.customer_id = p.customer_id) AS owner_matches,
            count(*) FILTER (WHERE c.customer_id <> p.customer_id) AS owner_mismatches
        FROM bronze.{table} c
        JOIN bronze.products p ON p.product_id = c.{product_field}
        WHERE c.{product_field} IS NOT NULL AND c.customer_id IS NOT NULL
    """)
    result.insert(0, "table", table)
    print(f"{table}:")
    display(result)

transactions:


,table,total_linked,owner_matches,owner_mismatches
0,transactions,4425008,4425008,0


digital_events:


,table,total_linked,owner_matches,owner_mismatches
0,digital_events,1094242,16,1094226


## 2. When a complaint's `customer_id` owns products themselves, does `affected_product_id` ever land on one of their own?

If `affected_product_id` were even loosely tied to the real customer (e.g. picking the wrong one of several), we'd expect at least some hits when the customer has multiple products. If it's fully independent of `customer_id`, we'd expect ~0% even here.

In [3]:
q("""
    WITH customers_with_products AS (
        SELECT DISTINCT customer_id FROM bronze.products
    )
    SELECT
        count(*) AS complaints_where_customer_owns_products,
        count(*) FILTER (WHERE c.affected_product_id IN (
            SELECT product_id FROM bronze.products p2 WHERE p2.customer_id = c.customer_id
        )) AS affected_product_is_actually_theirs
    FROM bronze.complaints c
    JOIN customers_with_products cwp ON cwp.customer_id = c.customer_id
    WHERE c.affected_product_id IS NOT NULL
""")

,complaints_where_customer_owns_products,affected_product_is_actually_theirs
0,41625,0


## 3. Temporal plausibility -- does the (wrong-owner) `affected_product_id` even exist as of the complaint's `creation_date`?

In [4]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE p.opening_date IS NOT NULL AND CAST(p.opening_date AS TIMESTAMP) > CAST(c.creation_date AS TIMESTAMP)) AS product_opened_after_complaint,
        count(*) FILTER (WHERE p.opening_date IS NOT NULL AND CAST(p.opening_date AS TIMESTAMP) <= CAST(c.creation_date AS TIMESTAMP)) AS product_existed_by_complaint
    FROM bronze.complaints c
    JOIN bronze.products p ON p.product_id = c.affected_product_id
    WHERE c.affected_product_id IS NOT NULL
""")

,total,product_opened_after_complaint,product_existed_by_complaint
0,44570,8351,36219


## 4. Is the mismatch structured or uniformly random?

Checking two things: (a) whether the SAME `affected_product_id` repeats suspiciously across many complaints (would suggest a fixed/default value rather than a real per-row assignment), and (b) whether the distribution of mismatched owners' `product_type`/`currency` looks like the full `products` population (uniform/random) or is skewed toward something specific.

In [5]:
print("Most repeated affected_product_id values (top 10):")
display(q("""
    SELECT affected_product_id, count(*) AS n
    FROM bronze.complaints WHERE affected_product_id IS NOT NULL
    GROUP BY 1 ORDER BY n DESC LIMIT 10
"""))
print()
print("Total distinct affected_product_id values used vs. total complaints with one populated:")
display(q("""
    SELECT count(*) AS complaints_with_product, count(DISTINCT affected_product_id) AS distinct_products_used
    FROM bronze.complaints WHERE affected_product_id IS NOT NULL
"""))

Most repeated affected_product_id values (top 10):


,affected_product_id,n
0,PRD-MHELT805RST3,5
1,PRD-U9TPURY8EMUZ,4
2,PRD-D59ISYJTWZAX,3
3,PRD-H0WDILTR1VDD,3
4,PRD-ISIIZBOYTIO8,3
5,PRD-V2S65NQJA7KE,3
6,PRD-34RM71MXQDAK,3
7,PRD-ILJ7GWX9I1PP,3
8,PRD-L4OH13R70RAD,3
9,PRD-D976R0MELVW1,3



Total distinct affected_product_id values used vs. total complaints with one populated:


,complaints_with_product,distinct_products_used
0,44570,42184


In [6]:
print("product_type distribution among affected products (complaints):")
display(q("""
    SELECT p.product_type, count(*) AS n
    FROM bronze.complaints c JOIN bronze.products p ON p.product_id = c.affected_product_id
    WHERE c.affected_product_id IS NOT NULL
    GROUP BY 1 ORDER BY n DESC
"""))
print()
print("product_type distribution across ALL of bronze.products, for comparison:")
display(q("SELECT product_type, count(*) AS n FROM bronze.products GROUP BY 1 ORDER BY n DESC"))

product_type distribution among affected products (complaints):


,product_type,n
0,Cuenta Ahorro,13517
1,Cuenta Corriente,11224
2,Tarjeta Crédito,11064
3,Tarjeta Débito,4422
4,Préstamo Personal,2169
5,Préstamo Hipotecario,1324
6,Inversión,633
7,Seguro,217



product_type distribution across ALL of bronze.products, for comparison:


,product_type,n
0,Cuenta Ahorro,120203
1,Tarjeta Crédito,100102
2,Cuenta Corriente,99979
3,Tarjeta Débito,39938
4,Préstamo Personal,19960
5,Préstamo Hipotecario,11910
6,Inversión,5859
7,Seguro,2049


## 5. Sample -- 15 mismatched rows, side by side

Complaint's own `customer_id`, the `affected_product_id`'s real owner, and both parties' basic info, to eyeball whether anything looks structurally related (e.g. same segment, same branch, sequential IDs) rather than fully random.

In [7]:
q("""
    SELECT
        c.complaint_id, c.customer_id AS complainant_customer_id, c.affected_product_id,
        p.customer_id AS actual_product_owner, p.product_type, p.currency,
        cust1.segment AS complainant_segment, cust2.segment AS owner_segment,
        cust1.registration_branch_id AS complainant_branch, cust2.registration_branch_id AS owner_branch
    FROM bronze.complaints c
    JOIN bronze.products p ON p.product_id = c.affected_product_id
    JOIN bronze.customers cust1 ON cust1.customer_id = c.customer_id
    JOIN bronze.customers cust2 ON cust2.customer_id = p.customer_id
    WHERE c.affected_product_id IS NOT NULL
    ORDER BY c.complaint_id
    LIMIT 15
""")

,complaint_id,complainant_customer_id,affected_product_id,actual_product_owner,product_type,currency,complainant_segment,owner_segment,complainant_branch,owner_branch
0,CMP-000P3Q3VK8DRP89MGDT8,CLI-C834EL7NBBVK,PRD-5SM6K5ZBM4VP,CLI-2C1B8GEYTYH3,Cuenta Ahorro,USD,Plus,Plus,SUC-UJOJRY9Z,SUC-C9HB36R0
1,CMP-0010RFZPRZG040O9UFUD,CLI-93BNDS6VKTVS,PRD-QWUHJ5Q6BV6A,CLI-BRKCLYQS8ER0,Cuenta Corriente,USD,Plus,Student,SUC-TPQWDM51,SUC-49UYREB4
2,CMP-00150MRGI6YS9FLRDLQD,CLI-I4NWNWN6Z76R,PRD-LKN8J7RFQKC7,CLI-K158ANTIHW1O,Cuenta Ahorro,USD,Basic,Basic,SUC-4Y7VCHAL,SUC-2G3S4P2J
3,CMP-001RCP3PJ6BT8SSTOIRO,CLI-ZZCA73T1AGXJ,PRD-96HGR1SNRKEG,CLI-C3OIHM4MNMBV,Cuenta Ahorro,ARS,Premium,Plus,SUC-L0WTDFDU,SUC-O1KZT3G1
4,CMP-005ODQTHQ8QJAQNSR9ZQ,CLI-EBNYFV9YOCXU,PRD-ZSXUPNC6ZNAM,CLI-YND5O6GAFLX8,Cuenta Ahorro,ARS,Basic,Basic,SUC-I12YRE1P,SUC-XAO0UODZ
5,CMP-007D4LERZ798ZBLZ778W,CLI-Q41KEXYPP367,PRD-QEO2EXDM1YVG,CLI-CWORMAJUH1TY,Préstamo Hipotecario,ARS,Plus,Basic,SUC-YVMZFHU6,SUC-8FH11H1Q
6,CMP-007MVWXSM50CZ1G6TAYH,CLI-J5P51M1I0N9P,PRD-NWI58MVLS99M,CLI-HWF4A7EEMD6E,Cuenta Corriente,USD,Basic,Basic,SUC-GSE3RF8H,SUC-AXCP89MN
7,CMP-008FKDHF1X4DBIIKWHSV,CLI-AQDL4KJF1NV9,PRD-J0MFEKJLSNLQ,CLI-SV9ZC3SN3SQI,Tarjeta Débito,ARS,Plus,Basic,SUC-FJ0Y8PQN,SUC-6EKVX6AW
8,CMP-008PL938VSBLT17TYBJH,CLI-PGQ8JNOODVAO,PRD-DLDSIBI9MA0J,CLI-VUBGL2K31DQU,Cuenta Ahorro,COP,Basic,Basic,SUC-NYKV5VYW,SUC-4C1NHWMP
9,CMP-008RSTOUVCAVP94FPW1L,CLI-9MFILIFHP7ZN,PRD-CEQQIPDJJRLI,CLI-5TLV2R1TLVRY,Tarjeta Débito,USD,Basic,Plus,SUC-MAGMMIR2,SUC-8NO0Y6IE


In [8]:
con.close()
print("closed")

closed
